# Training MobileNetV2 on CIFAR-10
MobileNetV2 is a popular neural network architecture known for its efficiency in mobile and edge devices. It introduces inverted residuals and linear bottlenecks. Here, we will implement and train a MobileNetV2 model on the CIFAR dataset.

## 1. Import Libraries

In [ ]:
_ROOT_ = '../'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

## 2. Define Hyperparameters
Set up the hyperparameters for the training process.

In [ ]:
batch_size = 128
num_epochs = 3
learning_rate = 0.01

## 3. Load and Transform the CIFAR-10 Dataset
Load the CIFAR dataset and apply necessary transformations.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root=_ROOT_ + './Data/cifar10', train=True, download=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root=_ROOT_ + './Data/cifar10', train=False, download=False, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


Visualisation of image samples from CIFAR-10 dataset.

![Architecture](./img/cifar10.png)

## 4. Define the MobileNetV2 Model

MobileNetV2 is an efficient neural network architecture primarily designed for mobile and resource-constrained environments. It was introduced by Google researchers and serves as an improvement over its predecessor, MobileNetV1, by introducing two significant architectural innovations: Inverted Residuals and Linear Bottlenecks. Here are the key features and design principles of MobileNetV2:

1. **Inverted Residual Structure:** Unlike traditional residual networks that add the input of a block to the output of layers within the block, MobileNetV2 introduces an inverted design. In this design, the input and output are typically thin (low-dimensional), and the representation inside the block is expanded, making the intermediate layer "thicker" (high-dimensional). This structure allows for efficient feature representations and reduces computational complexity.

2. **Linear Bottlenecks:** MobileNetV2 uses linear bottlenecks at the end of each block. This design choice is based on the observation that non-linearities like ReLU can destroy information, especially when used at low-dimensional representations. By using linear layers (without non-linear activation functions) at bottlenecks, the network preserves more information, especially during the transition from high to low-dimensional feature spaces.

3. **Depthwise Separable Convolutions:** Similar to MobileNetV1, MobileNetV2 employs depthwise separable convolutions, which factorize a standard convolution into a depthwise convolution and a pointwise convolution. This factorization significantly reduces the computational cost and the number of parameters while maintaining similar or better performance.

4. **Efficient and Compact:** The architecture is designed to be very efficient in terms of both the number of parameters and the computational cost, making it suitable for mobile devices, embedded systems, and other applications where computational resources are limited.

5. **Versatile Applicability:** Despite its compact and efficient design, MobileNetV2 delivers competitive performance on a variety of tasks and benchmarks, including image classification, object detection, and semantic segmentation.

6. **Streamlined for On-Device AI:** MobileNetV2 is not just a theoretical concept; it is practically oriented towards on-device AI applications. Its efficiency and performance make it suitable for real-time applications on smartphones and other portable devices.


### The following diagrams illustrate the structure of the MobileNetV2 network. The first diagram showcases the individual network module of MobileNetV2, highlighting its internal architecture. The second diagram presents the overall structure of the entire MobileNetV2 model, detailing how each module is interconnected to form the complete network.

![Architecture](./img/mobilenet_1.png) ![Architecture](./img/mobilenet_2.png) 

### Create MoblieNetV2 Architecture

In [ ]:
def _make_divisible(v, divisor, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_planes, out_planes, kernel_size=3, stride=1, groups=1, norm_layer=nn.BatchNorm2d):
        padding = (kernel_size - 1) // 2
        super(ConvBNReLU, self).__init__(
            nn.Conv2d(in_planes, out_planes, kernel_size, stride, padding, groups=groups, bias=False),
            norm_layer(out_planes),
            nn.ReLU6(inplace=True)
        )


class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio, norm_layer=nn.BatchNorm2d):
        super(InvertedResidual, self).__init__()
        self.stride = stride
        hidden_dim = int(round(inp * expand_ratio))
        self.use_res_connect = self.stride == 1 and inp == oup

        layers = []
        if expand_ratio != 1:
            # pw
            layers.append(ConvBNReLU(inp, hidden_dim, kernel_size=1, norm_layer=norm_layer))
        layers.extend([
            # dw
            ConvBNReLU(hidden_dim, hidden_dim, stride=stride, groups=hidden_dim, norm_layer=norm_layer),
            # pw-linear
            nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
            norm_layer(oup),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class MobileNetV2(nn.Module):
    def __init__(self,
                 num_classes=10,
                 width_mult=1.0,
                 inverted_residual_setting=None,
                 round_nearest=8,
                 block=InvertedResidual,
                 norm_layer=nn.BatchNorm2d,
                 dropout=0.2,
                 **kwargs):
        """
        MobileNet V2 main class
        Args:
            num_classes (int): Number of classes
            width_mult (float): Width multiplier - adjusts number of channels in each layer by this amount
            inverted_residual_setting: Network structure
            round_nearest (int): Round the number of channels in each layer to be a multiple of this number
            Set to 1 to turn off rounding
            block: Module specifying inverted residual building block for mobilenet
            norm_layer: Module specifying the normalization layer to use
        """
        super(MobileNetV2, self).__init__()

        input_channel = 32
        last_channel = 1280

        if inverted_residual_setting is None:
            inverted_residual_setting = [
                # t, c, n, s
                [1, 16, 1, 1],
                [6, 24, 2, 2],
                [6, 32, 3, 2],
                [6, 64, 4, 2],
                [6, 96, 3, 1],
                [6, 160, 3, 2],
                [6, 320, 1, 1],
            ]

        # only check the first element, assuming user knows t,c,n,s are required
        if len(inverted_residual_setting) == 0 or len(inverted_residual_setting[0]) != 4:
            raise ValueError("inverted_residual_setting should be non-empty "
                             "or a 4-element list, got {}".format(inverted_residual_setting))

        # building first layer
        input_channel = _make_divisible(input_channel * width_mult, round_nearest)
        self.last_channel = _make_divisible(last_channel * max(1.0, width_mult), round_nearest)
        features = [ConvBNReLU(3, input_channel, stride=2, norm_layer=norm_layer)]
        # building inverted residual blocks
        for t, c, n, s in inverted_residual_setting:
            output_channel = _make_divisible(c * width_mult, round_nearest)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(block(input_channel, output_channel, stride, expand_ratio=t, norm_layer=norm_layer))
                input_channel = output_channel
        # building last several layers
        features.append(ConvBNReLU(input_channel, self.last_channel, kernel_size=1, norm_layer=norm_layer))
        # make it nn.Sequential
        self.features = nn.Sequential(*features)
        # building classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.last_channel, num_classes),
        )

        # weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def _forward_impl(self, x):
        # This exists since TorchScript doesn't support inheritance, so the superclass method
        # (this one) needs to have a name other than `forward` that can be accessed in a subclass
        x = self.features(x)
        # Cannot use "squeeze" as batch-size can be 1 => must use reshape with x.shape[0]
        x = nn.functional.adaptive_avg_pool2d(x, 1).reshape(x.shape[0], -1)
        x = self.classifier(x)
        return x

    def forward(self, x):
        return self._forward_impl(x)


## 5. Define a Loss Function and Optimizer

In [ ]:
import torch.optim as optim

model = MobileNetV2()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## 6. Train the Network

In [ ]:
for epoch in range(num_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 0:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 100))
            running_loss = 0.0

print('Finished Training')

## 7. Save the Model
Save the trained model for future use.

In [ ]:
torch.save(model.state_dict(), './mobilenet_v2.pth')

## 8.Test the Model
Evaluate the model's performance on the test dataset.

In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the 10000 test images: {100 * correct / total} %')

## 9. Visualisation the result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load the model
model = MobileNetV2()
model.load_state_dict(torch.load(_ROOT_ + './Model/mobilenet-epoch_050.pth'))
model.eval()

# Choose one figure from Validation dataset
dataiter = iter(test_loader)
images, labels = next(dataiter)
idx = np.random.randint(images.shape[0])  # random choose one index
img, label = images[idx], labels[idx]

# Image Validation
img = (img * 0.5 + 0.5).clamp(0, 1) # Denomalise the image
img_np = img.numpy()
plt.imshow(np.transpose(img_np, (1, 2, 0)))
plt.title(f'Actual label: {label}')
plt.show()

# Predict the image
with torch.no_grad():
    img = img.unsqueeze(0)
    outputs = model(img)
    _, predicted = torch.max(outputs, 1)

# CIFAR-10 classes
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Print the result of prediction
print(f'Predicted: {classes[predicted[0]]}, Actual: {classes[label]}')


## References:
1. CIFAR-100 Dataset: Krizhevsky, A., & Hinton, G. (2009). ["Learning multiple layers of features from tiny images"](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf).

2. PyTorch Documentation: Accessible at [PyTorch Docs](https://pytorch.org/docs/stable/index.html).

3. Torchvision Models: Detailed in the [Torchvision Models Documentation](https://pytorch.org/vision/stable/models.html).

4. PyTorch Vision Models:[PyTorch Vision Model Zoo](https://pytorch.org/vision/stable/models.html)


